# 🏘️ 04_wui_cdp_calculation.ipynb
================================================================================

**Script 04 of 08** in the Networks Paper analysis pipeline.

## 🎯 Purpose
Calculate Wildland-Urban Interface (WUI) land cover statistics for each CDP. This tells us what type of environment surrounds each community - critical for understanding wildfire exposure.

## 🔥 Why This Matters for Wildfire Research
The WUI is where homes meet or intermix with wildland vegetation - the areas most at risk from wildfires. By classifying each CDP's land cover, we can:
- **Identify high-risk communities** surrounded by flammable vegetation
- **Correlate egress capacity** with WUI exposure
- **Prioritize** communities that have BOTH limited egress AND high WUI exposure

## 🗺️ What is WUI?
**Wildland-Urban Interface (WUI)** has two types:
- **Intermix WUI**: Houses scattered among wildland vegetation
- **Interface WUI**: Houses adjacent to large blocks of wildland vegetation

Both types face elevated wildfire risk compared to purely urban or rural areas.

## 📋 Workflow
1. **Load CDP shapefiles** from all states (step 01 output)
2. **Load WUI raster** (national 30m resolution classification)
3. **Align coordinate systems** (reproject CDPs to match raster)
4. **Compute zonal statistics** (count pixels of each WUI class per CDP)
5. **Rename columns** from numeric codes to descriptive class names
6. **Export** results to CSV

---

## 📥 INPUTS (Required Data Sources)
| Source | Path | Description |
|--------|------|-------------|
| CDP Shapefiles | `.../us-census-designated-places/{State}_{FIPS}/tl_2023_{FIPS}_place.shp` | Place boundaries (from step 01) |
| WUI Raster (VRT) | `.../wilderness-urban-interface/raw/NA/mosaic/WUI.vrt` | National WUI classification |

### 🏷️ WUI Class Definitions
| Value | Class Name | Description |
|-------|------------|-------------|
| 0 | Non-WUI: Water | Lakes, rivers, ocean |
| 1 | Forest/Shrub Intermix WUI | Homes scattered in forest/shrub |
| 2 | Forest/Shrub Interface WUI | Homes adjacent to forest/shrub |
| 3 | Grassland Intermix WUI | Homes scattered in grassland |
| 4 | Grassland Interface WUI | Homes adjacent to grassland |
| 5 | Non-WUI: Forest/Shrub | Wildland with no homes |
| 6 | Non-WUI: Grassland | Grassland with no homes |
| 7 | Non-WUI: Urban | Dense urban (low fire risk) |
| 8 | Non-WUI: Other | Agriculture, barren, etc. |

## 📤 OUTPUTS (Generated Files)
| Output | Path | Description |
|--------|------|-------------|
| WUI Land Cover CSV | `.../output_csvs/wui_land_cover_cdp/wui_land_cover_cdp.csv` | Pixel counts per WUI class per CDP |

## ⏱️ Expected Runtime
- ~1-2 hours for all CDPs (parallel processing with 300 cores)

---

## ⚙️ Settings

In [ ]:
################################################################################
#                            📦 IMPORT LIBRARIES                               #
################################################################################

import os
import geopandas as gpd
from rasterstats import zonal_stats
import rasterio
from time import time
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from tqdm import tqdm
import pandas as pd
import glob
from datetime import datetime
import pytz

################################################################################
#                            🔧 SETTINGS & CONFIGURATION                       #
################################################################################

# =============================================================================
# ⏰ Timezone Configuration (Pacific Time for timestamps)
# =============================================================================
TIMEZONE = pytz.timezone('America/Los_Angeles')

def print_timestamped(message):
    """Print message with Pacific Time timestamp."""
    timestamp = datetime.now(TIMEZONE).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"[{timestamp}] {message}")

# =============================================================================
# 🖥️ Parallel Processing Settings
# =============================================================================
NUM_CORES = 300  # Number of cores for parallel zonal statistics processing

################################################################################
#                            📂 FILE PATHS                                      #
################################################################################
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Configure BASE_DATA_DIR to point to your root data directory.          │
# │  Also set WUI_VRT_PATH to point to your WUI raster (.vrt) file.        │
# │  Override either with environment variables:                            │
# │      export NETWORKS_DATA_DIR="/your/path/to/data"                      │
# │      export WUI_VRT_PATH="/your/path/to/WUI.vrt"                       │
# │  Or edit the default paths below.                                       │
# └─────────────────────────────────────────────────────────────────────────┘
BASE_DATA_DIR = os.environ.get("NETWORKS_DATA_DIR", os.path.expanduser("~/data/networks_paper"))

# =============================================================================
# 📥 INPUT PATHS
# =============================================================================
# Path to the WUI mosaic VRT file (obtain from SILVIS Lab: https://silvis.forest.wisc.edu/data/wui-change/)
vrt_file_path = os.environ.get("WUI_VRT_PATH", os.path.expanduser("~/data/wui/WUI.vrt"))
cdp_parent_dir = os.path.join(BASE_DATA_DIR, "us-census-designated-places")

# =============================================================================
# 📤 OUTPUT PATHS
# =============================================================================
output_dir = os.path.join(BASE_DATA_DIR, "output_csvs", "wui_land_cover_cdp")
output_csv_path = os.path.join(output_dir, "wui_land_cover_cdp.csv")

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

################################################################################
#                            🛠️ HELPER FUNCTIONS                               #
################################################################################
def process_polygon(args):
    geoid, poly, vrt_path = args
    try:
        single_stats = zonal_stats(
            poly,
            vrt_path,
            categorical=True,
            nodata=None,
            all_touched=True,
            geojson_out=True
        )
        # Add the GEOID to link back to the shapefile
        single_stats[0]['properties']['GEOID'] = geoid
        return single_stats[0]
    except Exception as e:
        print(f"Error processing polygon GEOID {geoid}: {e}")
        return None

################################################################################
#                            📥 LOAD CDP SHAPEFILES                            #
################################################################################
print_timestamped("📂 Loading all CDP shapefiles from each state and DC...")
shapefile_pattern = os.path.join(cdp_parent_dir, "*", "tl_2023_*_place.shp")
shapefile_paths = glob.glob(shapefile_pattern)

if not shapefile_paths:
    raise FileNotFoundError(f"No shapefiles found with pattern {shapefile_pattern}")

print(f"Found {len(shapefile_paths)} shapefiles. Loading...")

gdf_list = []
for shp_path in shapefile_paths:
    try:
        print(f"Loading shapefile: {shp_path}")
        gdf = gpd.read_file(shp_path)
        gdf_list.append(gdf)
        print(f"Loaded {len(gdf)} polygons.")
    except Exception as e:
        print(f"Error loading {shp_path}: {e}")

if not gdf_list:
    raise ValueError("No shapefiles were loaded successfully.")

cdp_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

# Check if GEOID exists in the data
if 'GEOID' not in cdp_gdf.columns:
    raise KeyError("GEOID column not found in the shapefiles.")

total_polygons = len(cdp_gdf)
print(f"All shapefiles loaded and concatenated. Total CDP polygons: {total_polygons}")

################################################################################
#                            🗺️ CRS ALIGNMENT                                   #
################################################################################
print("Checking and aligning CRS...")
with rasterio.open(vrt_file_path) as src:
    raster_crs = src.crs

cdp_gdf = cdp_gdf.to_crs(raster_crs)
print("CDP polygons reprojected to raster CRS:", raster_crs)

################################################################################
#                            📐 PREPARE POLYGONS                               #
################################################################################
print("Extracting polygon geometries for parallel processing...")
polygons = list(cdp_gdf["geometry"])
geoids = list(cdp_gdf["GEOID"])
indexed_polygons = list(zip(geoids, polygons))

################################################################################
#                            🖥️ PARALLEL PROCESSING SETUP                      #
################################################################################
available_cores = multiprocessing.cpu_count()
num_cores = min(NUM_CORES, available_cores)
print(f"Using {num_cores} cores out of {available_cores} available.")

################################################################################
#                            📊 COMPUTE ZONAL STATISTICS                       #
################################################################################
print(f"Starting parallel zonal statistics computation using {num_cores} cores...")
start_time = time()

results = []
tasks = [(geoid, poly, vrt_file_path) for geoid, poly in indexed_polygons]

with ProcessPoolExecutor(max_workers=num_cores) as executor:
    for result in tqdm(executor.map(process_polygon, tasks), total=total_polygons, desc="Processing Polygons"):
        if result is not None:
            results.append(result)

print("All polygons processed in parallel.")
print(f"Elapsed time: {time() - start_time:.2f}s")

################################################################################
#                            🔄 CONVERT RESULTS                                #
################################################################################
print("Converting accumulated results to GeoDataFrame...")
result_gdf = gpd.GeoDataFrame.from_features(results, crs=raster_crs)

################################################################################
#                            🏷️ RENAME WUI CLASS COLUMNS                       #
################################################################################
wui_classes = {
    0: "Non-WUI: Water",
    1: "Forest/Shrubland/Wetland-dominated Intermix WUI",
    2: "Forest/Shrubland/Wetland-dominated Interface WUI",
    3: "Grassland-dominated Intermix WUI",
    4: "Grassland-dominated Interface WUI",
    5: "Non-WUI: Forest/Shrub/Wetland-dominated",
    6: "Non-WUI: Grassland-dominated",
    7: "Non-WUI: Urban",
    8: "Non-WUI: Other"
}

col_rename_map = {}
for col in result_gdf.columns:
    try:
        col_int = int(col)
        if col_int in wui_classes:
            col_rename_map[col] = wui_classes[col_int]
    except (ValueError, TypeError):
        pass

result_gdf = result_gdf.rename(columns=col_rename_map)

################################################################################
#                            🧹 CLEAN UP DATA                                  #
################################################################################
if 'geometry' in result_gdf.columns:
    result_gdf = result_gdf.drop(columns='geometry')

# Fill NA and empty values with zeros
# Replace all NaN values with 0
result_gdf.fillna(0, inplace=True)
# Replace empty strings or whitespace-only strings with 0
result_gdf.replace(r'^\s*$', 0, regex=True, inplace=True)

################################################################################
#                            💾 SAVE RESULTS                                   #
################################################################################
print("Final DataFrame columns after renaming and dropping geometry:")
print(result_gdf.columns.tolist())

print("\nHead of the final DataFrame:")
print(result_gdf.head())

result_gdf.to_csv(output_csv_path, index=False)
print(f"Final zonal statistics data saved to: {output_csv_path}")
print(f"Process completed. Total time: {time() - start_time:.2f}s")


In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# ── Configure path ─────────────────────────────────────────────────────────────
BASE_DATA_DIR = os.environ.get("NETWORKS_DATA_DIR", os.path.expanduser("~/data/networks_paper"))

# ---------------------------
# 1. Read and process CSV data
# ---------------------------
csv_file_path = os.path.join(BASE_DATA_DIR, "output_csvs", "combined_csv", "combined_data.csv")
df = pd.read_csv(csv_file_path)

# Ensure Total Popu is numeric and filter out non-positive populations
df['Total Popu'] = pd.to_numeric(df['Total Popu'], errors='coerce')
df = df[df['Total Popu'] > 0]

# ---------------------------
# 2. Compute “exits” variable
# ---------------------------
# Sum the five boundary crossing edges columns and divide by 2.
exit_cols = [
    'boundary_crossing_edges_motorway',
    'boundary_crossing_edges_trunk',
    'boundary_crossing_edges_primary',
    'boundary_crossing_edges_secondary',
    'boundary_crossing_edges_tertiary'
]
df['exits'] = df[exit_cols].sum(axis=1, skipna=True) / 2

# ---------------------------
# 3. Create binned variables
# ---------------------------
# Bin exits into four categories.
exit_bins = [0, 1, 6, 10, np.inf]
# Note: The original R code uses labels: Very Few, Few, Medium, Many.
# However, for the bivariate plot we want the “exit” dimension to increase from left (low) to right (high).
# So here we keep the same labels, but later in the legend we will reverse the order.
exit_labels = ["Very Few Lanes", "Few Lanes", "Medium Lanes", "Many Lanes"]
df['exit_bins'] = pd.cut(df['exits'], bins=exit_bins, labels=exit_labels, include_lowest=True)

# Bin burn probability (bp_mean) into four categories.
bp_bins = [0, 0.001, 0.005, 0.01, np.inf]
bp_labels = ["Low Burn", "Medium Burn", "High Burn", "Very High Burn"]
df['bp_mean_bins'] = pd.cut(df['bp_mean'], bins=bp_bins, labels=bp_labels, include_lowest=True)

# Create the bivariate bin by combining the two binned variables.
# For the bivariate mapping, we want the “exit” dimension to be in descending order (i.e. “Many Lanes” first)
# even though pd.cut returns factors in ascending order. We therefore later set the categorical ordering.
df['bivariate_bins'] = df['exit_bins'].astype(str) + "." + df['bp_mean_bins'].astype(str)

# Define the desired ordering for the bivariate bins.
bivariate_order = [
    "Many Lanes.Low Burn", "Medium Lanes.Low Burn", "Few Lanes.Low Burn", "Very Few Lanes.Low Burn",
    "Many Lanes.Medium Burn", "Medium Lanes.Medium Burn", "Few Lanes.Medium Burn", "Very Few Lanes.Medium Burn",
    "Many Lanes.High Burn", "Medium Lanes.High Burn", "Few Lanes.High Burn", "Very Few Lanes.High Burn",
    "Many Lanes.Very High Burn", "Medium Lanes.Very High Burn", "Few Lanes.Very High Burn", "Very Few Lanes.Very High Burn"
]
df['bivariate_bins'] = pd.Categorical(df['bivariate_bins'], categories=bivariate_order, ordered=True)

# ---------------------------
# 4. Create a GeoDataFrame for the points
# ---------------------------
# Use INTPTLON and INTPTLAT as point coordinates (assumed to be in EPSG:4326).
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.INTPTLON, df.INTPTLAT), crs="EPSG:4326")

# ---------------------------
# 5. Download and reproject U.S. state polygons
# ---------------------------
# Download state boundaries from the Census Bureau TIGER/Line shapefile.
states_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip"
states = gpd.read_file(states_url)

# Reproject both datasets to a US-focused projection (NAD83 / Conus Albers, EPSG:5070)
gdf = gdf.to_crs("EPSG:5070")
states = states.to_crs("EPSG:5070")

# ---------------------------
# 6. Define a bivariate color scheme
# ---------------------------
# For the bivariate color scheme, we assign weights to each category.
# Here we want the “exit” dimension to increase from left to right:
#   "Very Few Lanes" -> 0.0, "Few Lanes" -> 0.33, "Medium Lanes" -> 0.66, "Many Lanes" -> 1.0.
# And for burn, from bottom to top:
#   "Low Burn" -> 0.0, "Medium Burn" -> 0.33, "High Burn" -> 0.66, "Very High Burn" -> 1.0.
exit_levels = ["Many Lanes", "Medium Lanes", "Few Lanes", "Very Few Lanes"]
burn_levels = ["Low Burn", "Medium Burn", "High Burn", "Very High Burn"]

exit_weights = {"Many Lanes": 1.0, "Medium Lanes": 0.66, "Few Lanes": 0.33, "Very Few Lanes": 0.0}
burn_weights = {"Low Burn": 0.0, "Medium Burn": 0.33, "High Burn": 0.66, "Very High Burn": 1.0}

# Use a sequential colormap; here we choose 'YlOrRd'.
cmap = cm.get_cmap("YlOrRd")

# Build the dictionary mapping bivariate bin labels to colors.
bivariate_colors = {}
for ex in exit_levels:
    for br in burn_levels:
        # Compute the combined weight as the average of the exit and burn weights.
        combined_weight = (exit_weights[ex] + burn_weights[br]) / 2
        bin_label = f"{ex}.{br}"
        bivariate_colors[bin_label] = mcolors.to_hex(cmap(combined_weight))

# ---------------------------
# 7. Plot the bivariate map
# ---------------------------
fig, ax = plt.subplots(figsize=(12, 8))

# Plot state boundaries.
states.boundary.plot(ax=ax, linewidth=1, edgecolor='black')

# Plot the points, mapping each point's bivariate bin to its color.
# (Markersize is set small for a dense plot.)
gdf.plot(ax=ax, color=gdf['bivariate_bins'].map(bivariate_colors), markersize=2)

ax.set_title("Bivariate Map of Exits and Burn Severity")
plt.show()

# ---------------------------
# 8. Create a bivariate legend (4x4 grid)
# ---------------------------
# For the legend, we want the x-axis (exit bins) to run from low to high:
legend_exit = ["Very Few Lanes", "Few Lanes", "Medium Lanes", "Many Lanes"]
# And the y-axis (burn bins) from low to high:
legend_burn = ["Low Burn", "Medium Burn", "High Burn", "Very High Burn"]

fig, ax = plt.subplots(figsize=(3, 3))

# Draw colored squares for each combination.
for i, ex in enumerate(legend_exit):
    for j, br in enumerate(legend_burn):
        # Note: In the bivariate_colors dictionary, keys are defined with exit levels in the order
        # ["Many Lanes", "Medium Lanes", "Few Lanes", "Very Few Lanes"].
        # So to get the proper color for a legend cell (with exit increasing left-to-right),
        # we need to look up the reversed exit level.
        # For legend, the leftmost cell corresponds to "Very Few Lanes".
        key = f"{ex}.{br}"
        color = bivariate_colors.get(key, "#ffffff")
        # Draw a rectangle at grid cell (i, j).
        ax.add_patch(plt.Rectangle((i, j), 1, 1, facecolor=color))

# Set grid limits and ticks.
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_xticks(np.arange(0.5, 4.5, 1))
ax.set_yticks(np.arange(0.5, 4.5, 1))
ax.set_xticklabels(legend_exit, rotation=45, fontsize=8)
ax.set_yticklabels(legend_burn, fontsize=8)
ax.set_xlabel("Exit Bins (Low → High)", fontsize=9)
ax.set_ylabel("Burn Severity (Low → High)", fontsize=9)
ax.set_title("Bivariate Legend", fontsize=10)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()
